# Part 1: Crawl Prime Lands

Crawl https://www.primelands.lk and save markdown files and a JSONL corpus.

In [ ]:
from prime_land.config import CONFIG, get_path
from prime_land.ingest_document.web_crawler import PrimeLandCrawler
from prime_land.ingest_document.writers import save_jsonl, save_markdown

settings = dict(CONFIG["crawling"])
base_url = settings.pop("base_url")
start_urls = [base_url + path for path in settings.pop("start_paths")]

In [ ]:
crawler = PrimeLandCrawler(base_url=base_url, **settings)
documents = await crawler.crawl_async(start_urls)
print(len(documents), "documents,", len(crawler.failed), "failed")

In [ ]:
markdown_count = save_markdown(documents, get_path("markdown_dir"))
corpus_count = save_jsonl(documents, get_path("corpus_file"))
print(markdown_count, "markdown files,", corpus_count, "corpus entries")

In [ ]:
import json

with open(get_path("corpus_file"), encoding="utf-8") as file:
    corpus = [json.loads(line) for line in file]

properties = [doc for doc in corpus if doc["page_type"] == "property"]
fields = ["property_id", "title", "address", "price", "bedrooms", "bathrooms", "sqft", "amenities", "agent"]

assert len(properties) >= 5, "Need at least 5 property entries"
for field in fields:
    filled = sum(1 for doc in properties if doc["metadata"].get(field))
    print(f"{field}: {filled}/{len(properties)}")